# All Baseline Models — PyTorch (Rice Disease Dataset)

**Run all cells top-to-bottom.**  
All models train from scratch (`weights=None`, all layers trainable).  
Weights + history are saved to `Model/` after each model.

| Cell | Content |
|------|---------|
| 1 | Imports & Device Setup |
| 2 | Configuration & Data Loading |
| 3 | Shared Utilities |
| 4 | **ResNet50** |
| 5 | **VGG16** |
| 6 | **MobileNetV2** |
| 7 | **InceptionV3** |
| 8 | Final Summary |

In [1]:
import os, copy, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'Device      : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'CUDA        : {torch.version.cuda}')
    print(f'Total VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')


PyTorch     : 2.5.1+cu121
Torchvision : 0.20.1+cu121
Device      : cuda
GPU         : NVIDIA RTX 5000 Ada Generation
CUDA        : 12.1
Total VRAM  : 31.99 GB


In [2]:
# ── Hyper-parameters ──────────────────────────────────────────
BATCH_SIZE  = 64
IMAGE_SIZE  = (224, 224)
EPOCHS      = 50
RANDOM_SEED = 42

DATA_DIR   = r'..\Rice_Disease_Dataset'
OUTPUT_DIR = 'Model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'Output directory : {os.path.abspath(OUTPUT_DIR)}')

# ── Discover classes ──────────────────────────────────────────
classes = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
NUM_CLASSES  = len(classes)
class_to_idx = {cls: i for i, cls in enumerate(classes)}

print('Classes:    ', classes)
print('num_classes:', NUM_CLASSES)

# ── Build filepath / label dataframe ─────────────────────────
filepaths, labels = [], []
for cls in classes:
    cls_dir = os.path.join(DATA_DIR, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            filepaths.append(os.path.join(cls_dir, f))
            labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print('Total Images:', len(df))

# ── 80 / 10 / 10 stratified split ────────────────────────────
df_train, df_temp = train_test_split(
    df, test_size=0.20, shuffle=True,
    stratify=df['label'], random_state=123
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, shuffle=True,
    stratify=df_temp['label'], random_state=123
)
print('Train:', len(df_train))
print('Val:  ', len(df_val))
print('Test: ', len(df_test))

# ── Class weights ─────────────────────────────────────────────
y_train_encoded  = df_train['label'].map(class_to_idx).values
class_weights    = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train_encoded
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print('Class weights:')
for cls, w in zip(classes, class_weights):
    print(f'  {cls:<25} : {w:.4f}')


Output directory : c:\Users\km1612\Documents\riceMobileNet\RiceMobileNet_final\RiceMobileNEt Code\Experiment_on_Multi_Source_DATASET\Baseline_Models\Model
Classes:     ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scaled', 'narrow_brown_spot']
num_classes: 6
Total Images: 14758
Train: 11806
Val:   1476
Test:  1476
Class weights:
  bacterial_leaf_blight     : 1.0383
  brown_spot                : 0.8195
  healthy                   : 1.0938
  leaf_blast                : 0.7428
  leaf_scaled               : 0.9760
  narrow_brown_spot         : 1.8811


In [3]:
# =============================================================================
#  SHARED UTILITIES  (run once — used by every model below)
# =============================================================================

# ── Custom Dataset ────────────────────────────────────────────
class ImageDataFrameDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.encoded   = [class_to_idx[l] for l in self.df['label']]
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.encoded[idx]


# ── Normalization constants ───────────────────────────────────
IMAGENET_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
MINUSONE_NORM = transforms.Normalize([0.5,   0.5,   0.5  ], [0.5,   0.5,   0.5  ])
IDENTITY_NORM = transforms.Normalize([0.0,   0.0,   0.0  ], [1.0,   1.0,   1.0  ])


# ── Early Stopping ────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=7):
        self.patience     = patience
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights = None
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss    = val_loss
            self.counter      = 0
            self.best_weights = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience
    def restore(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)


# ── DataLoader factory ────────────────────────────────────────
def make_loaders(train_t, val_t, batch_size=BATCH_SIZE):
    pin = (DEVICE.type == 'cuda')
    train_loader = DataLoader(
        ImageDataFrameDataset(df_train, train_t),
        batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=pin)
    val_loader = DataLoader(
        ImageDataFrameDataset(df_val, val_t),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    test_loader = DataLoader(
        ImageDataFrameDataset(df_test, val_t),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    return train_loader, val_loader, test_loader


# ── Keras-style progress bar ──────────────────────────────────
def _bar(current, total, width=30):
    filled = int(width * current / total)
    return '=' * filled + '-' * (width - filled)


# ── Training Engine ───────────────────────────────────────────
def train_model(model, model_name, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    es     = EarlyStopping(patience=7)

    history = {k: [] for k in [
        'loss', 'accuracy', 'precision', 'recall', 'f1',
        'val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'val_f1'
    ]}

    n_batches = len(train_loader)

    print(f'\nModel  : {model_name}')
    print(f'Train batches : {n_batches}  |  Val batches : {len(val_loader)}')
    print(f'Device : {DEVICE}' + (
        f'  ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
    print('-' * 110)

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──
        model.train()
        t0 = time.time()
        ep_loss = 0.0
        ep_preds, ep_labels = [], []

        for bi, (imgs, labels) in enumerate(train_loader, 1):
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                out = model(imgs)
                if isinstance(out, torchvision.models.inception.InceptionOutputs):
                    out = out.logits
                loss = criterion(out, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            ep_loss += loss.item() * imgs.size(0)
            ep_preds.extend(out.argmax(1).detach().cpu().numpy())
            ep_labels.extend(labels.detach().cpu().numpy())

            print(f'\rEpoch {epoch}/{EPOCHS} {bi}/{n_batches} [{_bar(bi, n_batches)}]',
                  end='', flush=True)

        # ── Train metrics ──
        tr_loss = ep_loss / len(train_loader.dataset)
        tr_acc  = accuracy_score(ep_labels, ep_preds)
        tr_prec = precision_score(ep_labels, ep_preds, average='weighted', zero_division=0)
        tr_rec  = recall_score(ep_labels,   ep_preds, average='weighted', zero_division=0)
        tr_f1   = f1_score(ep_labels,       ep_preds, average='weighted', zero_division=0)

        # ── Validate ──
        model.eval()
        vl_loss = 0.0
        vl_preds, vl_labels = [], []

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(DEVICE,   non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                    out = model(imgs)
                    if isinstance(out, torchvision.models.inception.InceptionOutputs):
                        out = out.logits
                    loss = criterion(out, labels)
                vl_loss += loss.item() * imgs.size(0)
                vl_preds.extend(out.argmax(1).cpu().numpy())
                vl_labels.extend(labels.cpu().numpy())

        vl_loss /= len(val_loader.dataset)
        vl_acc   = accuracy_score(vl_labels, vl_preds)
        vl_prec  = precision_score(vl_labels, vl_preds, average='weighted', zero_division=0)
        vl_rec   = recall_score(vl_labels,   vl_preds, average='weighted', zero_division=0)
        vl_f1    = f1_score(vl_labels,       vl_preds, average='weighted', zero_division=0)

        elapsed   = time.time() - t0
        step_time = elapsed / n_batches

        print(
            f'\rEpoch {epoch}/{EPOCHS} {n_batches}/{n_batches} [{"=" * 30}] - '
            f'{elapsed:.0f}s {step_time:.1f}s/step - '
            f'loss: {tr_loss:.4f} - accuracy: {tr_acc:.4f} - '
            f'precision: {tr_prec:.4f} - recall: {tr_rec:.4f} - f1: {tr_f1:.4f} - '
            f'val_loss: {vl_loss:.4f} - val_accuracy: {vl_acc:.4f} - '
            f'val_precision: {vl_prec:.4f} - val_recall: {vl_rec:.4f} - val_f1: {vl_f1:.4f}',
            flush=True
        )
        print()

        for key, val in [
            ('loss',          tr_loss), ('accuracy',      tr_acc),
            ('precision',     tr_prec), ('recall',        tr_rec),  ('f1',      tr_f1),
            ('val_loss',      vl_loss), ('val_accuracy',  vl_acc),
            ('val_precision', vl_prec), ('val_recall',    vl_rec),  ('val_f1',  vl_f1),
        ]:
            history[key].append(round(float(val), 6))

        scheduler.step(vl_loss)

        if es(vl_loss, model):
            print(f'  >> Early stopping at epoch {epoch}. '
                  f'Best val_loss = {es.best_loss:.4f}')
            break

    es.restore(model)
    return history


# ── Test Evaluation ───────────────────────────────────────────
def evaluate_on_test(model, test_loader):
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(DEVICE), label_smoothing=0.1)
    model.eval()
    tl, tp, tg = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(DEVICE,   non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                out = model(imgs)
                if isinstance(out, torchvision.models.inception.InceptionOutputs):
                    out = out.logits
                loss = criterion(out, labels)
            tl += loss.item() * imgs.size(0)
            tp.extend(out.argmax(1).cpu().numpy())
            tg.extend(labels.cpu().numpy())
    tl   /= len(test_loader.dataset)
    acc   = accuracy_score(tg, tp)
    prec  = precision_score(tg, tp, average='weighted', zero_division=0)
    rec   = recall_score(tg,   tp, average='weighted', zero_division=0)
    f1    = f1_score(tg,       tp, average='weighted', zero_division=0)
    print(f'\n{"=" * 60}')
    print('  TEST EVALUATION')
    print(f'{"=" * 60}')
    print(f'  loss      : {tl:.4f}')
    print(f'  accuracy  : {acc:.4f}')
    print(f'  precision : {prec:.4f}')
    print(f'  recall    : {rec:.4f}')
    print(f'  f1_score  : {f1:.4f}')
    print(f'{"=" * 60}')
    print()
    print('Classification Report:')
    print(classification_report(tg, tp, target_names=classes, digits=4))
    return tp, tg


print('All shared utilities ready.')

# ── Model Statistics Helper ──────────────────────────────────────────────────
def compute_model_stats(model, model_name, input_size=(1, 3, 224, 224), n_runs=100):
    """
    Returns a dict with:
      total_params_M      : total parameters (millions)
      trainable_params_M  : trainable parameters (millions)
      disk_size_mb        : .pth file size on disk (MB) — estimated from param bytes
      flops_g             : FLOPs (GFLOPs) via torch.profiler
      inference_ms        : mean single-image inference time (ms)
      fps                 : frames per second
    """
    import os, tempfile, time
    # ── Param counts ──────────────────────────────────────────
    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ── Disk size: save state_dict to a temp file, measure bytes ──
    tmp = tempfile.NamedTemporaryFile(suffix='.pth', delete=False)
    tmp.close()
    torch.save(model.state_dict(), tmp.name)
    disk_mb = os.path.getsize(tmp.name) / 1024**2
    os.unlink(tmp.name)

    # ── FLOPs via torch.profiler CPU-op counting ──────────────
    flops_g = None
    try:
        from torch.profiler import profile, ProfilerActivity, record_function
        dummy = torch.randn(*input_size).to(DEVICE)
        model.eval()
        with torch.no_grad():
            with profile(activities=[ProfilerActivity.CPU],
                         record_shapes=True,
                         with_flops=True) as prof:
                with record_function("model_inference"):
                    out = model(dummy)
        total_flops = sum(e.flops for e in prof.key_averages() if e.flops > 0)
        flops_g = total_flops / 1e9
    except Exception:
        flops_g = float('nan')

    # ── Inference latency ─────────────────────────────────────
    model.eval()
    dummy = torch.randn(*input_size).to(DEVICE)
    # warm-up
    with torch.no_grad():
        for _ in range(10):
            _ = model(dummy)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            _ = model(dummy)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    inf_ms = (sum(times) / len(times)) * 1000
    fps    = 1000.0 / inf_ms

    stats = {
        'model'               : model_name,
        'total_params_M'      : round(total_p     / 1e6, 4),
        'trainable_params_M'  : round(trainable_p / 1e6, 4),
        'disk_size_mb'        : round(disk_mb,            4),
        'flops_g'             : round(flops_g,            4) if flops_g == flops_g else None,
        'inference_ms'        : round(inf_ms,             4),
        'fps'                 : round(fps,                2),
    }

    print(f'\n  ── Model Statistics: {model_name} ──')
    print(f'  Total Params (M)      : {stats["total_params_M"]}')
    print(f'  Trainable Params (M)  : {stats["trainable_params_M"]}')
    print(f'  Disk Size (MB)        : {stats["disk_size_mb"]}')
    print(f'  FLOPs (G)             : {stats["flops_g"]}')
    print(f'  Inference (ms)        : {stats["inference_ms"]}')
    print(f'  FPS                   : {stats["fps"]}')
    return stats



# ── Save all results to model-specific subfolder ─────────────────────────────
def save_all_results(model, model_name, history, model_stats,
                     test_preds, test_labels, input_size=(1, 3, 224, 224)):
    """
    Saves everything for one model into OUTPUT_DIR/<model_name>/:
      <model_name>_weights.pth
      <model_name>_history.json
      <model_name>_stats.json          (params, disk size, FLOPs, latency, FPS)
      <model_name>_test_metrics.json   (loss, acc, prec, rec, f1)
      <model_name>_classification_report.txt
    """
    import os
    folder = os.path.join(OUTPUT_DIR, model_name)
    os.makedirs(folder, exist_ok=True)

    # weights
    w_path = os.path.join(folder, f'{model_name}_weights.pth')
    torch.save(model.state_dict(), w_path)

    # history
    h_path = os.path.join(folder, f'{model_name}_history.json')
    with open(h_path, 'w') as f:
        json.dump(history, f, indent=2)

    # model stats
    s_path = os.path.join(folder, f'{model_name}_stats.json')
    with open(s_path, 'w') as f:
        json.dump(model_stats, f, indent=2)

    # test metrics
    from sklearn.metrics import (accuracy_score, precision_score,
                                  recall_score, f1_score, classification_report)
    test_metrics = {
        'accuracy' : round(accuracy_score(test_labels, test_preds),  6),
        'precision': round(precision_score(test_labels, test_preds,
                                           average='weighted', zero_division=0), 6),
        'recall'   : round(recall_score(test_labels, test_preds,
                                        average='weighted', zero_division=0), 6),
        'f1'       : round(f1_score(test_labels, test_preds,
                                    average='weighted', zero_division=0), 6),
    }
    tm_path = os.path.join(folder, f'{model_name}_test_metrics.json')
    with open(tm_path, 'w') as f:
        json.dump(test_metrics, f, indent=2)

    # classification report
    cr = classification_report(test_labels, test_preds,
                                target_names=classes, digits=4)
    cr_path = os.path.join(folder, f'{model_name}_classification_report.txt')
    with open(cr_path, 'w') as f:
        f.write(cr)

    print(f'  All results saved -> {folder}/')
    print(f'    {os.path.basename(w_path)}')
    print(f'    {os.path.basename(h_path)}')
    print(f'    {os.path.basename(s_path)}')
    print(f'    {os.path.basename(tm_path)}')
    print(f'    {os.path.basename(cr_path)}')


print('All shared utilities ready.')


All shared utilities ready.
All shared utilities ready.


In [4]:
# =============================================================================
#  MODEL 1 : ResNet50
# =============================================================================
print('=' * 70)
print('  MODEL: ResNet50')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
resnet_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    IMAGENET_NORM,
])
resnet_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    IMAGENET_NORM,
])

resnet_train_loader, resnet_val_loader, resnet_test_loader = make_loaders(resnet_train_t, resnet_val_t)

# ── Architecture ──────────────────────────────────────────────
# weights=None  → random initialisation, train from scratch
# base_model.trainable = True  → all layers are trainable (PyTorch default)
def build_resnet50(num_classes=NUM_CLASSES):
    model = torchvision.models.resnet50(weights=None)  # no ImageNet weights
    for p in model.parameters():                       # all layers trainable
        p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

resnet_model = build_resnet50(NUM_CLASSES).to(DEVICE)
print(f'Trainable params : {sum(p.numel() for p in resnet_model.parameters() if p.requires_grad):,}')

resnet_history = train_model(resnet_model, 'ResNet50', resnet_train_loader, resnet_val_loader)
resnet_test_preds, resnet_test_labels = evaluate_on_test(resnet_model, resnet_test_loader)
resnet_stats = compute_model_stats(resnet_model, 'ResNet50')
save_all_results(resnet_model, 'ResNet50', resnet_history, resnet_stats,
                 resnet_test_preds, resnet_test_labels)

del resnet_model, resnet_train_loader, resnet_val_loader, resnet_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nResNet50 done.\n')


  MODEL: ResNet50
Trainable params : 23,520,326

Model  : ResNet50
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 103s 0.6s/step - loss: 1.7707 - accuracy: 0.2497 - precision: 0.2717 - recall: 0.2497 - f1: 0.2516 - val_loss: 1.7315 - val_accuracy: 0.3543 - val_precision: 0.3821 - val_recall: 0.3543 - val_f1: 0.3366

Epoch 2/50 185/185 [==============================] - 46s 0.2s/step - loss: 1.6770 - accuracy: 0.3655 - precision: 0.3756 - recall: 0.3655 - f1: 0.3612 - val_loss: 1.6239 - val_accuracy: 0.4329 - val_precision: 0.4545 - val_recall: 0.4329 - val_f1: 0.4143

Epoch 3/50 185/185 [==============================] - 46s 0.2s/step - loss: 1.5196 - accuracy: 0.4631 - precision: 0.4818 - recall: 0.4631 - f1: 0.4546 - val_loss: 1.4548 - val_accuracy: 0.5305 - val_precision: 0.5776 

In [5]:
# =============================================================================
#  MODEL 2 : VGG16
# =============================================================================
print('=' * 70)
print('  MODEL: VGG16')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
vgg_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    IMAGENET_NORM,
])
vgg_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    IMAGENET_NORM,
])

vgg_train_loader, vgg_val_loader, vgg_test_loader = make_loaders(vgg_train_t, vgg_val_t)

# ── Architecture ──────────────────────────────────────────────
# weights=None  → random initialisation, train from scratch
# base_model.trainable = True  → all layers are trainable (PyTorch default)
# VGG16 features output: 512ch x 7x7 for 224x224 input
# AdaptiveAvgPool2d(1) = GlobalAveragePooling2D → 512-dim vector
def build_vgg16(num_classes=NUM_CLASSES):
    model = torchvision.models.vgg16(weights=None)  # no ImageNet weights
    for p in model.parameters():                    # all layers trainable
        p.requires_grad = True
    model.avgpool    = nn.AdaptiveAvgPool2d(1)
    model.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(512, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.5),
        nn.Linear(512, num_classes),
    )
    return model

vgg_model = build_vgg16(NUM_CLASSES).to(DEVICE)
print(f'Trainable params : {sum(p.numel() for p in vgg_model.parameters() if p.requires_grad):,}')

vgg_history = train_model(vgg_model, 'VGG16', vgg_train_loader, vgg_val_loader)
vgg_test_preds, vgg_test_labels = evaluate_on_test(vgg_model, vgg_test_loader)
vgg_stats = compute_model_stats(vgg_model, 'VGG16')
save_all_results(vgg_model, 'VGG16', vgg_history, vgg_stats,
                 vgg_test_preds, vgg_test_labels)

del vgg_model, vgg_train_loader, vgg_val_loader, vgg_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nVGG16 done.\n')


  MODEL: VGG16
Trainable params : 14,980,422

Model  : VGG16
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 139s 0.8s/step - loss: 1.7142 - accuracy: 0.2567 - precision: 0.2883 - recall: 0.2567 - f1: 0.2460 - val_loss: 1.4951 - val_accuracy: 0.4485 - val_precision: 0.5195 - val_recall: 0.4485 - val_f1: 0.4519

Epoch 2/50 185/185 [==============================] - 121s 0.7s/step - loss: 1.4391 - accuracy: 0.4545 - precision: 0.4652 - recall: 0.4545 - f1: 0.4509 - val_loss: 1.3887 - val_accuracy: 0.5346 - val_precision: 0.5683 - val_recall: 0.5346 - val_f1: 0.5092

Epoch 3/50 185/185 [==============================] - 116s 0.6s/step - loss: 1.3206 - accuracy: 0.5545 - precision: 0.5631 - recall: 0.5545 - f1: 0.5548 - val_loss: 1.2638 - val_accuracy: 0.5955 - val_precision: 0.6201 - va

In [6]:
# =============================================================================
#  MODEL 3 : MobileNetV2
# =============================================================================
print('=' * 70)
print('  MODEL: MobileNetV2')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
mobilenet_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    MINUSONE_NORM,
])
mobilenet_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    MINUSONE_NORM,
])

mobilenet_train_loader, mobilenet_val_loader, mobilenet_test_loader = make_loaders(
    mobilenet_train_t, mobilenet_val_t
)

# ── Architecture ──────────────────────────────────────────────
# weights=None  → random initialisation, train from scratch
# base_model.trainable = True  → all layers are trainable (PyTorch default)
# torchvision MobileNetV2: features output 1280ch; built-in AdaptiveAvgPool2d(1)
def build_mobilenetv2(num_classes=NUM_CLASSES):
    model = torchvision.models.mobilenet_v2(weights=None)  # no ImageNet weights
    for p in model.parameters():                           # all layers trainable
        p.requires_grad = True
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, num_classes),
    )
    return model

mobilenet_model = build_mobilenetv2(NUM_CLASSES).to(DEVICE)
print(f'Trainable params : {sum(p.numel() for p in mobilenet_model.parameters() if p.requires_grad):,}')

mobilenet_history = train_model(
    mobilenet_model, 'MobileNetV2', mobilenet_train_loader, mobilenet_val_loader
)
mobilenet_test_preds, mobilenet_test_labels = evaluate_on_test(mobilenet_model, mobilenet_test_loader)
mobilenet_stats = compute_model_stats(mobilenet_model, 'MobileNetV2')
save_all_results(mobilenet_model, 'MobileNetV2', mobilenet_history, mobilenet_stats,
                 mobilenet_test_preds, mobilenet_test_labels)

del mobilenet_model, mobilenet_train_loader, mobilenet_val_loader, mobilenet_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nMobileNetV2 done.\n')


  MODEL: MobileNetV2
Trainable params : 2,231,558

Model  : MobileNetV2
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 39s 0.2s/step - loss: 1.8059 - accuracy: 0.1988 - precision: 0.2046 - recall: 0.1988 - f1: 0.1937 - val_loss: 1.7671 - val_accuracy: 0.2134 - val_precision: 0.2230 - val_recall: 0.2134 - val_f1: 0.1607

Epoch 2/50 185/185 [==============================] - 34s 0.2s/step - loss: 1.7780 - accuracy: 0.2133 - precision: 0.2313 - recall: 0.2133 - f1: 0.2053 - val_loss: 1.7459 - val_accuracy: 0.2263 - val_precision: 0.2250 - val_recall: 0.2263 - val_f1: 0.1614

Epoch 3/50 185/185 [==============================] - 34s 0.2s/step - loss: 1.7534 - accuracy: 0.2299 - precision: 0.2537 - recall: 0.2299 - f1: 0.2224 - val_loss: 1.7039 - val_accuracy: 0.2947 - val_precision: 0.3

In [7]:
# =============================================================================
#  MODEL 4 : InceptionV3
# =============================================================================
print('=' * 70)
print('  MODEL: InceptionV3')
print('=' * 70)

# ── Transforms ────────────────────────────────────────────────
inception_train_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.2, 0.2),
        scale=(0.7, 1.3),
        shear=8.5,
    ),
    transforms.ColorJitter(brightness=0.3),
    transforms.ToTensor(),
    MINUSONE_NORM,
])
inception_val_t = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    MINUSONE_NORM,
])

inception_train_loader, inception_val_loader, inception_test_loader = make_loaders(
    inception_train_t, inception_val_t
)

# ── Architecture ──────────────────────────────────────────────
# weights=None  → random initialisation, train from scratch
# base_model.trainable = True  → all layers are trainable (PyTorch default)
# aux_logits=False: avoids the auxiliary branch which crashes on 224x224 inputs
# when training from scratch (no pretrained checkpoint needed).
def build_inceptionv3(num_classes=NUM_CLASSES):
    model = torchvision.models.inception_v3(
        weights=None,       # no ImageNet weights
        aux_logits=False    # disable aux branch — safe without pretrained weights
    )
    for p in model.parameters():  # all layers trainable
        p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

inception_model = build_inceptionv3(NUM_CLASSES).to(DEVICE)
print(f'Trainable params : {sum(p.numel() for p in inception_model.parameters() if p.requires_grad):,}')

inception_history = train_model(
    inception_model, 'InceptionV3', inception_train_loader, inception_val_loader
)
inception_test_preds, inception_test_labels = evaluate_on_test(inception_model, inception_test_loader)
inception_stats = compute_model_stats(inception_model, 'InceptionV3')
save_all_results(inception_model, 'InceptionV3', inception_history, inception_stats,
                 inception_test_preds, inception_test_labels)

del inception_model, inception_train_loader, inception_val_loader, inception_test_loader
if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print('\nInceptionV3 done.\n')


  MODEL: InceptionV3
Trainable params : 21,797,862

Model  : InceptionV3
Train batches : 185  |  Val batches : 24
Device : cuda  (NVIDIA RTX 5000 Ada Generation)
--------------------------------------------------------------------------------------------------------------
Epoch 1/50 185/185 [==============================] - 46s 0.2s/step - loss: 1.8035 - accuracy: 0.2084 - precision: 0.2349 - recall: 0.2084 - f1: 0.2129 - val_loss: 1.7202 - val_accuracy: 0.3523 - val_precision: 0.4069 - val_recall: 0.3523 - val_f1: 0.3296

Epoch 2/50 185/185 [==============================] - 44s 0.2s/step - loss: 1.7406 - accuracy: 0.2835 - precision: 0.3019 - recall: 0.2835 - f1: 0.2760 - val_loss: 1.6237 - val_accuracy: 0.4404 - val_precision: 0.4881 - val_recall: 0.4404 - val_f1: 0.4282

Epoch 3/50 185/185 [==============================] - 44s 0.2s/step - loss: 1.6481 - accuracy: 0.3563 - precision: 0.3723 - recall: 0.3563 - f1: 0.3443 - val_loss: 1.5099 - val_accuracy: 0.4851 - val_precision: 0.

In [8]:
# =============================================================================
#  TRAINING COMPLETE — all results saved under Model/<ModelName>/
# =============================================================================
import glob as _glob

print('=' * 70)
print('  ALL MODELS TRAINING COMPLETE')
print('=' * 70)
print()
print('Saved files (model-wise folders):')
for p in sorted(_glob.glob(os.path.join(OUTPUT_DIR, '**', '*'), recursive=True)):
    if os.path.isfile(p):
        size_kb = os.path.getsize(p) / 1024
        rel = os.path.relpath(p, OUTPUT_DIR)
        print(f'  {rel:<65}  ({size_kb:.1f} KB)')

print()
print('Each model folder contains:')
print('  <ModelName>_weights.pth               — model weights')
print('  <ModelName>_history.json              — train/val metrics per epoch')
print('  <ModelName>_stats.json                — params, disk size, FLOPs, latency, FPS')
print('  <ModelName>_test_metrics.json         — test accuracy, precision, recall, F1')
print('  <ModelName>_classification_report.txt — per-class report')
print()
print('-- How to reload weights (example: ResNet50) --')
print("  model = build_resnet50(NUM_CLASSES)")
print("  model.load_state_dict(torch.load('Model/ResNet50/ResNet50_weights.pth', map_location=DEVICE))")
print("  model.to(DEVICE).eval()")
print()
print('-- How to reload history (example: ResNet50) --')
print("  with open('Model/ResNet50/ResNet50_history.json') as f:")
print("      h = json.load(f)")
print("  # h['val_accuracy'], h['val_f1'], h['loss'], ...")


  ALL MODELS TRAINING COMPLETE

Saved files (model-wise folders):
  InceptionV3\InceptionV3_classification_report.txt                  (0.6 KB)
  InceptionV3\InceptionV3_history.json                               (7.5 KB)
  InceptionV3\InceptionV3_stats.json                                 (0.2 KB)
  InceptionV3\InceptionV3_test_metrics.json                          (0.1 KB)
  InceptionV3\InceptionV3_weights.pth                                (85480.6 KB)
  MobileNetV2\MobileNetV2_classification_report.txt                  (0.6 KB)
  MobileNetV2\MobileNetV2_history.json                               (7.5 KB)
  MobileNetV2\MobileNetV2_stats.json                                 (0.2 KB)
  MobileNetV2\MobileNetV2_test_metrics.json                          (0.1 KB)
  MobileNetV2\MobileNetV2_weights.pth                                (8959.3 KB)
  ResNet50\ResNet50_classification_report.txt                        (0.6 KB)
  ResNet50\ResNet50_history.json                                     